Calendar Features 

In [4]:
import pandas as pd
import numpy as np

# Đọc dữ liệu từ bước 3
df = pd.read_csv('train_bi.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['ItemCode', 'Date']).reset_index(drop=True)

# Tạo các biến thời gian cơ bản
df['dayofweek'] = df['Date'].dt.dayofweek
df['month'] = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x in [5, 6] else 0)

Lag Features

In [ ]:
# Tạo các biến trễ (Shift) an toàn cho dự báo chuỗi 56 ngày
lags = [56, 57, 58, 63, 70, 77, 84] # 56 (đúng mốc), 63 (cách 1 tuần), 70 (cách 2 tuần)...

for lag in lags:
    df[f'lag_{lag}'] = df.groupby('ItemCode')['Quantity'].shift(lag)

Rolling Window Features

In [ ]:
# Chọn các cửa sổ thời gian trượt: 7 ngày, 14 ngày, 28 ngày, 56 ngày
windows = [7, 14, 28, 56]

for w in windows:
    # Trung bình trượt dựa trên lag_56
    df[f'rolling_mean_{w}'] = df.groupby('ItemCode')['lag_56'].transform(lambda x: x.rolling(w).mean())
    
    # Trung vị trượt (Trị outlier chọc trời)
    df[f'rolling_median_{w}'] = df.groupby('ItemCode')['lag_56'].transform(lambda x: x.rolling(w).median())
    
    # Độ lệch chuẩn (Đo lường mức độ biến động ổn định hay thất thường của SKU)
    df[f'rolling_std_{w}'] = df.groupby('ItemCode')['lag_56'].transform(lambda x: x.rolling(w).std())

Static Features

In [5]:
import time

print("--- Bắt đầu xử lý Static Features (Bản tối ưu siêu tốc) ---")
start_time = time.time()

# 1. Đọc file dữ liệu gốc (Chỉ lấy 3 cột cần thiết để tiết kiệm RAM)
df_raw = pd.read_csv('ket_qua.csv', usecols=['ItemCode', 'Quantity', 'SalesAmount'])

# 2. TẠO TRƯỚC CÁC CỘT PHỤ BẰNG VECTORIZATION (Thay thế hoàn toàn cho hàm lambda cũ)
# Ép toàn bộ số âm về 0 để chỉ giữ lại lượng xuất bán thực tế
df_raw['Qty_Out'] = df_raw['Quantity'].clip(lower=0)
# Ép toàn bộ số dương về 0 để chỉ giữ lại lượng bị khách trả (mang dấu âm)
df_raw['Qty_In'] = df_raw['Quantity'].clip(upper=0)

# 3. GOM NHÓM THEO SKU (Chỉ dùng hàm 'sum' gốc của Pandas nên tốc độ cực nhanh)
sku_stats = df_raw.groupby('ItemCode').agg(
    Total_Sales=('SalesAmount', 'sum'),
    Total_Qty_Out=('Qty_Out', 'sum'),
    Total_Qty_In=('Qty_In', 'sum')
).reset_index()

# 4. TÍNH TOÁN RETURN RATE (Tỷ lệ trả hàng)
# Lấy trị tuyệt đối của lượng trả chia cho lượng bán. Nếu mã nào chưa từng bị trả (mẫu số = 0), điền số 0.
sku_stats['return_rate'] = (sku_stats['Total_Qty_In'].abs() / sku_stats['Total_Qty_Out']).fillna(0)

# 5. PHÂN LOẠI ABC CLASS DỰA TRÊN TỔNG DOANH THU (Total_Sales)
# Sắp xếp giảm dần theo doanh thu để tính phần trăm tích lũy
sku_stats = sku_stats.sort_values(by='Total_Sales', ascending=False)
sku_stats['cum_percentage'] = sku_stats['Total_Sales'].cumsum() / sku_stats['Total_Sales'].sum() * 100

# Hàm phân loại mã hóa thành số (0, 1, 2) để mô hình LightGBM/XGBoost đọc được trực tiếp
def assign_abc(pct):
    if pct <= 80: return 0    # Nhóm A: Gánh 80% doanh thu (Bán chạy nhất)
    elif pct <= 95: return 1  # Nhóm B: Gánh 15% doanh thu tiếp theo
    else: return 2            # Nhóm C: Long-tail (Nhóm bán chậm, thưa thớt)

sku_stats['ABC_Class'] = sku_stats['cum_percentage'].apply(assign_abc)

# 6. TRÍCH XUẤT BẢNG ĐẶC TRƯNG TĨNH SẠCH
static_features = sku_stats[['ItemCode', 'ABC_Class', 'return_rate', 'Total_Sales']]

end_time = time.time()
print(f"--- Hoàn thành! Thời gian xử lý: {end_time - start_time:.2f} giây ---")

# Hiển thị thử 5 dòng đầu tiên để kiểm tra kết quả
print("\nXem thử dữ liệu Static Features vừa tạo:")
print(static_features.head())\
# Lưu bảng đặc trưng tĩnh ra file CSV để sử dụng cho bước tiếp theo
static_features.to_csv('static_features.csv', index=False)

--- Bắt đầu xử lý Static Features (Bản tối ưu siêu tốc) ---
--- Hoàn thành! Thời gian xử lý: 0.37 giây ---

Xem thử dữ liệu Static Features vừa tạo:
        ItemCode  ABC_Class  return_rate  Total_Sales
2      SKU-00003          0     0.000000  16697884148
9197   SKU-09458          0     0.000048  10081348060
11873  SKU-12176          0     0.020243   9272614773
1      SKU-00002          0     0.000000   8012686409
9492   SKU-09760          0     0.008480   5919637907


In [10]:
import time 
start_time = time.time()
#đọc dữ liệu train_bi.csv
df = pd.read_csv('train_bi.csv')
df['Date'] = pd.to_datetime(df['Date'])
#sắp xếp dữ liệu đồng nhất theo ItemCode và Date tăng dần để đảm bảo tính đúng đắn khi tạo các biến trễ
df = df.sort_values(['ItemCode', 'Date']).reset_index(drop=True)
#ghép cặp đặc trưng tĩnh vào dữ liệu train_bi
static_features = pd.read_csv('static_features.csv')
df = df.merge(static_features, on='ItemCode', how='left')
end_time = time.time()
print(df.head(10))

        Date   ItemCode  Quantity  ABC_Class  return_rate  Total_Sales
0 2020-11-17  SKU-00001         0          1          0.0     35414413
1 2020-11-18  SKU-00001         0          1          0.0     35414413
2 2020-11-19  SKU-00001         0          1          0.0     35414413
3 2020-11-20  SKU-00001         0          1          0.0     35414413
4 2020-11-21  SKU-00001         0          1          0.0     35414413
5 2020-11-22  SKU-00001         0          1          0.0     35414413
6 2020-11-23  SKU-00001         0          1          0.0     35414413
7 2020-11-24  SKU-00001         0          1          0.0     35414413
8 2020-11-25  SKU-00001         0          1          0.0     35414413
9 2020-11-26  SKU-00001         0          1          0.0     35414413


In [13]:
#Tạo biến thời gian lịch 
df['dayofweek'] = df['Date'].dt.dayofweek
df['month'] = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x in [5, 6] else 0)


In [14]:
#tạo biến trễ (LAG) sử dụng groupby 
lags = [56, 57, 58, 63, 70, 77, 84] # 56 (đúng mốc), 63 (cách 1 tuần), 70 (cách 2 tuần)...
for lag in lags:
    df[f'lag_{lag}'] = df.groupby('ItemCode')['Quantity'].shift(lag)
end_time = time.time()


In [16]:
#Tạo biến thống kê trượt (Rolling) dựa trên lag_56
windows = [7, 14, 28, 56]
for w in windows:
    # Trung bình trượt dựa trên lag_56
    df[f'rolling_mean_{w}'] = df.groupby('ItemCode')['lag_56'].transform(lambda x: x.rolling(w).mean())
    
    # Trung vị trượt (Trị outlier chọc trời)
    df[f'rolling_median_{w}'] = df.groupby('ItemCode')['lag_56'].transform(lambda x: x.rolling(w).median())
    
    # Độ lệch chuẩn (Đo lường mức độ biến động ổn định hay thất thường của SKU)
    df[f'rolling_std_{w}'] = df.groupby('ItemCode')['lag_56'].transform(lambda x: x.rolling(w).std())
end_time = time.time()


In [17]:
df = df.dropna() # Loại bỏ các dòng có giá trị NaN do tạo biến trễ và thống kê trượt
print(df.head(10))

          Date   ItemCode  Quantity  ABC_Class  return_rate  Total_Sales  \
111 2021-03-08  SKU-00001         0          1          0.0     35414413   
112 2021-03-09  SKU-00001         0          1          0.0     35414413   
113 2021-03-10  SKU-00001         0          1          0.0     35414413   
114 2021-03-11  SKU-00001         0          1          0.0     35414413   
115 2021-03-12  SKU-00001         0          1          0.0     35414413   
116 2021-03-13  SKU-00001         0          1          0.0     35414413   
117 2021-03-14  SKU-00001         0          1          0.0     35414413   
118 2021-03-15  SKU-00001         0          1          0.0     35414413   
119 2021-03-16  SKU-00001         0          1          0.0     35414413   
120 2021-03-17  SKU-00001         0          1          0.0     35414413   

     dayofweek  month  quarter  is_weekend  ...  rolling_std_7  \
111          0      3        1           0  ...            0.0   
112          1      3        1 

In [5]:
df = pd.read_csv('train_features.csv')


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [9]:
#xóa các hàng có năm bé hơn 2022 để tập trung vào dữ liệu gần đây nhất, phản ánh xu hướng hiện tại của thị trường
df = df[df['Date'].dt.year >= 2022].reset_index(drop=True)
print(df.head(10))

        Date   ItemCode  Quantity  ABC_Class  return_rate  Total_Sales  \
0 2022-01-01  SKU-00001         0          1          0.0     35414413   
1 2022-01-02  SKU-00001         0          1          0.0     35414413   
2 2022-01-03  SKU-00001         0          1          0.0     35414413   
3 2022-01-04  SKU-00001         0          1          0.0     35414413   
4 2022-01-05  SKU-00001         0          1          0.0     35414413   
5 2022-01-06  SKU-00001         0          1          0.0     35414413   
6 2022-01-07  SKU-00001         0          1          0.0     35414413   
7 2022-01-08  SKU-00001         0          1          0.0     35414413   
8 2022-01-09  SKU-00001         0          1          0.0     35414413   
9 2022-01-10  SKU-00001         0          1          0.0     35414413   

   dayofweek  month  quarter  is_weekend  ...  rolling_std_7  rolling_mean_14  \
0          5      1        1           1  ...            0.0              0.0   
1          6      1    

In [10]:
#lưu file df ra final_data để sử dụng cho bước tiếp theo
df.to_csv('final_data.csv', index=False)